# Reconstruction Impact：用三个问题阅读 Raw/SVC

本 Notebook 是针对一个已声明 Raw/SVC 样本的可读执行路径。计算阶段仍按 `STAGE_ORDER` 展开，并与批量运行器和静态报告使用同一套已保存产物协议。

科学阅读按三个问题推进：

1. **重建后呈现什么，两侧差异在哪里？** 先读输入提供的 SVC 重建状态标签，再看 Raw 独立表达 baseline 和两侧各自的空间结果。
2. **State 与差异出现在哪里？** 先看 SVC 标签多样性和共同有效窗口上的 ΔNeff，再把这些窗口放回完整 Raw Anatomy；State 与 Gain 是不同的证据视图。
3. **位置与分子/成员有什么关系？** Moran 是两侧原生空间图上的平行分支；AUCell 先在固定单位和基因轴上评分，再做空间聚合；membership 只比较明确的共同 ID；集成表只连接已保存事实，不构成独立验证。

输入的 SVC 重建标签就是重建状态标签。表达资源缺失、阈值不稳定和可选阶段失败都会保留在结果中。样本声明缺失或无效时会在加载阶段直接报错；Notebook 不会悄悄替换成 example 样本。


## 输入与参数协议

默认的真实阅读对象是 `data/P2CRC_Xenium/sample.yaml`。如需分析其他已声明样本，可设置 `SAMPLE_YAML`；如需更换输出位置，可设置 `OUTPUT_DIR`。样本声明提供重建影响分析的默认参数。`RECONSTRUCTION_IMPACT_OVERRIDES` 接受 JSON 映射，用于明确的单次运行覆盖；执行前会显示最终参数和随机种子。

表达声明决定哪些消费者可运行。非负线性浮点值（包括很小的值）是合法输入，不因数值大小被四舍五入或静默裁剪；流程按声明的 scale 处理。流程不会猜测表达尺度、替换输入的重建标签，也不会修改任一输入对象。意外的阶段错误会继续抛出，以保留执行失败的 traceback；正常的科学前提不足则由流程记录为不可用。


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

# 只定位仓库，不静默替换用户请求的样本。
REPO_ROOT = next(
    candidate for candidate in (Path.cwd(), Path.cwd().parent)
    if (candidate / "revise_analysis").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    from IPython.display import Image, Markdown, display
except Exception:
    Image = None
    Markdown = None
    display = print

from revise_analysis.io import load_sample
from revise_analysis.analyses.reconstruction_impact import (
    ImpactWorkflow,
    STAGE_ORDER,
    effective_parameters,
)

DEFAULT_SAMPLE = REPO_ROOT / "data" / "P2CRC_Xenium" / "sample.yaml"
SAMPLE_YAML = Path(os.environ.get("SAMPLE_YAML", str(DEFAULT_SAMPLE))).expanduser()
if not SAMPLE_YAML.is_absolute():
    SAMPLE_YAML = (REPO_ROOT / SAMPLE_YAML).resolve()

OVERRIDE_TEXT = os.environ.get("RECONSTRUCTION_IMPACT_OVERRIDES", "{}")
OVERRIDES = json.loads(OVERRIDE_TEXT)
if not isinstance(OVERRIDES, dict):
    raise ValueError("RECONSTRUCTION_IMPACT_OVERRIDES must be a JSON object")

display(pd.DataFrame([
    {"参数": "SAMPLE_YAML", "值": str(SAMPLE_YAML)},
    {"参数": "OUTPUT_DIR", "值": os.environ.get("OUTPUT_DIR", "<加载样本后推导>")},
    {"参数": "RECONSTRUCTION_IMPACT_OVERRIDES", "值": OVERRIDE_TEXT},
    {"参数": "continue_on_error", "值": False},
]))


## 加载并验证已声明样本

加载是有意设置的硬边界。样本必须在 SVC 中包含配置指定的重建状态标签列，且标签值完整。这样可以防止缺失的 SVC 标签被静默替换成新的 Leiden 分群或备用 fixture。


In [ ]:
sample = load_sample(SAMPLE_YAML)

DEFAULT_OUTPUT = REPO_ROOT / "output" / "notebook" / sample.sample_id
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", str(DEFAULT_OUTPUT))).expanduser()
if not OUTPUT_DIR.is_absolute():
    OUTPUT_DIR = (REPO_ROOT / OUTPUT_DIR).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIGURED_PARAMETERS = dict(
    sample.config.get("analysis_parameters", {}).get("reconstruction_impact", {}) or {}
)
EFFECTIVE_PARAMETERS = effective_parameters(sample, OVERRIDES)
workflow = ImpactWorkflow(
    sample,
    OUTPUT_DIR,
    parameters=OVERRIDES,
    continue_on_error=False,
)

parameter_rows = []
for name, value in EFFECTIVE_PARAMETERS.items():
    parameter_rows.append({
        "参数": name,
        "配置值": CONFIGURED_PARAMETERS.get(name, "<包默认值>"),
        "单次覆盖": OVERRIDES.get(name, ""),
        "最终值": value,
    })
display(Markdown(
    f"已加载 {sample.sample_id!r}；配置的 SVC 重建标签列为 {sample.reconstruction_key!r}。"
))
display(pd.DataFrame(parameter_rows))


## 执行有序阶段

下面每个 Notebook 小节都通过 `run_stage` 调用一个命名的 `ImpactWorkflow` 阶段，然后读取当前结果清单。执行顺序保留为：输入、baseline、支持诊断、多样性、区域、Anatomy、分子结果、membership、集成和已保存结果图。三个问题只改变阅读转换，不改变阶段顺序、有效参数或计算对象。意外错误会继续传播；缺少科学前提会明确保留在当前结果中。


In [ ]:
result = workflow.result()

def run_visible_stage(name):
    global result
    record = workflow.run_stage(name)
    result = workflow.result()
    display(pd.DataFrame([{"阶段": record.get("stage"), "状态": record.get("status"), "产物": ", ".join(record.get("artifacts", []))}]))
    return record

display(pd.DataFrame([
    {"顺序": index, "阶段": name}
    for index, name in enumerate(STAGE_ORDER, start=1)
]))


## 按阶段阅读已保存产物

下面的辅助函数只读取刚刚保存到 `OUTPUT_DIR` 下、且列在当前 `result.outputs` 中的文件。这个 manifest 是 Notebook 的产物边界：目录里的旧文件不会被猜测为当前结果。当某个阶段只有部分结果时，Notebook 仍能保持可读，并为每个小节显示对应的产物路径。


In [ ]:
def saved_path(relative):
    # 当前结果清单是旧产物的边界。
    manifest = set(result.get("outputs", {}).values())
    if relative not in manifest:
        return None
    path = OUTPUT_DIR / relative
    return path if path.is_file() else None

def show_saved_table(relative, rows=12):
    path = saved_path(relative)
    if path is None:
        return False
    print(relative)
    if path.suffix.lower() == ".csv":
        display(pd.read_csv(path).head(rows))
    elif path.suffix.lower() == ".json":
        display(pd.DataFrame([json.loads(path.read_text(encoding="utf-8"))]))
    elif path.suffix.lower() in {".png", ".jpg", ".jpeg", ".svg"} and Image is not None:
        display(Image(filename=str(path)))
    else:
        print(f"已保存文件：{path}")
    return True

def show_section(section_id, rows=12):
    section = next(
        (item for item in result.get("sections", []) if item.get("id") == section_id),
        {"id": section_id, "artifacts": []},
    )
    display(Markdown(f"### {section.get('title', section_id)}"))
    if section.get("description"):
        display(Markdown(section["description"]))
    artifacts = section.get("artifacts", [])
    for relative in artifacts:
        show_saved_table(relative, rows=rows)
    if not artifacts:
        display(Markdown("本阶段没有保存产物。"))


## 问题一：重建后呈现什么，两侧差异在哪里？

输入小节记录单位数、基因数以及输入的重建状态标签计数；这些标签是 State 的对象，不由 Notebook 重新命名。baseline 小节分别标出 Raw Leiden 和已有 Raw Level2 证据。支持曲线先说明候选尺度下的 SVC parent、Raw 和共同有效窗口支持，但不使用 State、Gain 或 program 结果反向选择尺度。


In [ ]:
run_visible_stage("input")
show_section("input")
run_visible_stage("baseline")
show_section("baseline")
run_visible_stage("support")
show_section("support")


## 问题二：State 与差异出现在哪里？

State 直接读取已保存的 SVC 重建状态标签列。窗口多样性表会在阈值判断之前展示，因此即使结果为 `no_stable_threshold`，连续场也不会被删除。下一步的 Gain 只在共同有效窗口中读取 SVC 与 Raw Leiden 的 ΔNeff；它不能把标签变化单独解释成生物学改善。


In [ ]:
run_visible_stage("diversity")
show_section("diversity")


## 问题二续：把差异放回空间与 Anatomy

State 区域是连续 SVC 多样性场经过阈值筛选后的视图。Gain 是共同有效窗口上相对于 Raw Leiden 的候选 ΔNeff；两者的支持、分母和阈值状态分别保留。阈值失败时仍保留连续表、阈值状态和 bootstrap 审计，不会把结果伪装成数值为零的区域。


In [ ]:
run_visible_stage("regions")
show_section("regions")


## 问题二续：Anatomy 是空间解释背景

Anatomy 在完整 Raw broad 标签上以自己声明的尺度独立建立。Raw 和 SVC 的观测点都按坐标落入 Anatomy 网格。parent 窗口保留实际观测点组成，包括并列主导和缺失覆盖；不使用 parent 中心点或重叠面积的捷径。Anatomy 解释 State/Gain 的位置，但不替代 State/Gain 的定义。


In [ ]:
run_visible_stage("anatomy")
show_section("anatomy")


## 问题三：位置与分子/成员有什么关系？

Moran 是与 State/Gain 平行的分支：Raw 与 SVC 各自在自己的原生坐标图上计算，不依赖 Region，也不因为某个 Region 再建图。AUCell 先对固定单位集、基因轴和 gene set 做一次单位级评分，再将已保存分数按 window、Anatomy 或 Region 聚合；聚合不会重复评分。只有显式开启共同 ID 选项时才会生成 membership，且只比较 Raw-defined cohort 与完整 SVC 标签的 shared IDs。集成结果把已保存的 State/Gain、Anatomy、分子和 membership 事实按明确分母连接起来；它不会把部分完成的阶段提升为独立验证。


In [ ]:
run_visible_stage("molecular")
show_section("molecular")
run_visible_stage("membership")
show_section("membership")
run_visible_stage("integration")
show_section("integration")


## 三个问题的限制与最终综合

最后这个视图会保留所有不可用组件和阶段错误。最终综合从保存后的 `result.json` 读取确定性阅读摘要，并链接到同一份静态报告；它不会重新打开任一输入对象，也不会新增评分或独立验证。


In [ ]:
run_visible_stage("figures")
result = workflow.result()

def _json_safe(value):
    if isinstance(value, dict):
        return {str(key): _json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if hasattr(value, "item"):
        return _json_safe(value.item())
    return value

# 保存当前清单，使静态报告可以独立审阅，不必重新执行 Notebook 或打开输入对象。
result_for_report = dict(result)
result_for_report.update(schema_version=1, sample_id=sample.sample_id, analysis="reconstruction_impact")
result_for_report["outputs"] = dict(result.get("outputs", {}))
result_for_report["outputs"]["report"] = "report.html"
(OUTPUT_DIR / "result.json").write_text(
    json.dumps(_json_safe(result_for_report), ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
from revise_analysis.reporting import render_report
from revise_analysis.reporting.impact_reading import reading_summary
report_path = render_report(OUTPUT_DIR)

for relative in sorted(result.get("outputs", {}).values()):
    if relative.startswith("figures/"):
        show_saved_table(relative)

# The final reading is derived from the same saved result that feeds report.html.
reading_items = reading_summary(OUTPUT_DIR, result_for_report)[:5]
reading_lines = []
for item in reading_items:
    text = str(item.get("text", "")).strip()
    anchor = str(item.get("anchor", "")).strip().lstrip("#")
    if not text:
        continue
    target = f"report.html#{anchor}" if anchor else "report.html"
    reading_lines.append(f"- {text} ([报告定位]({target}))")
if reading_lines:
    display(Markdown("### 三个问题的综合阅读\n\n" + "\n".join(reading_lines)))

unavailable_rows = result.get("unavailable", [])
error_rows = result.get("stage_errors", [])
if unavailable_rows:
    display(Markdown("### 当前不可用组件"))
    display(pd.DataFrame(unavailable_rows))
if error_rows:
    display(Markdown("### 阶段执行错误"))
    display(pd.DataFrame(error_rows)[
        [column for column in ("stage", "error_type", "message")
         if column in pd.DataFrame(error_rows).columns]
    ])
file_index = pd.DataFrame([
    {"输出键": key, "保存路径": relative}
    for key, relative in sorted(result_for_report["outputs"].items())
])
display(Markdown(f"静态报告已保存到：[{report_path.name}]({report_path.name})"))
display(Markdown("### 已保存文件索引"))
display(file_index)
